# Pools bronze layer

### Imports

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

### Parameters

In [0]:
dbutils.widgets.text("entity_name", "pools")
dbutils.widgets.text("load_pattern", "2")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Variables

In [0]:
PAGE_SIZE = 1000
pools_list = []
query_variables = {
    "skip": 0
}

In [0]:
pools_query = """
  query PoolsQuery($skip: Int!){
    pools(first: 1000,
      skip: $skip, 
      orderBy: createdAtTimestamp,
      orderDirection: asc
    ) {
      id
      token0 { 
        symbol
        id
        decimals
      }
      token1 { 
        symbol
        id
        decimals
      }
      totalValueLockedToken0
      totalValueLockedToken1
      totalValueLockedETH
      totalValueLockedUSD
      volumeToken0
      volumeToken1
      volumeUSD
      token0Price
      token1Price
      feesUSD
      feeTier
      liquidity
      sqrtPrice
      tick
      txCount
      createdAtTimestamp
      createdAtBlockNumber
    }
  }
  """

### Execution

In [0]:
while True:
    print(f"*INFO*: Skip rows amount: {query_variables["skip"]}.")
    response = requests.post(SUBGRAPH_URL, json={"query": pools_query, "variables": query_variables})
    pools_list.extend(response.json()["data"][entity_name])

    time.sleep(0.3)

    rows_loaded = len(response.json()["data"][entity_name])
    print(f"*INFO*: Number of rows loaded: {rows_loaded}.")

    if rows_loaded == PAGE_SIZE:
        query_variables.update({"skip": query_variables["skip"] + PAGE_SIZE}) 
    elif rows_loaded == 0: 
        break
    else:
        query_variables.update({"skip": query_variables["skip"] + rows_loaded})     

In [0]:
pools_df = spark.createDataFrame(pools_list)
pools_df = pools_df.withColumn("load_timestamp", current_timestamp())

In [0]:
display(pools_df.count())

In [0]:
if load_pattern == "1":
    print("*INFO*: Executing full load.")
    pools_df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "2":
    print("*INFO*: Executing incremental load.")
    pools_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "3":
    print("*INFO*: Executing differential load.")
    old_pools_table = DeltaTable.forName(spark, f"uniswap.bronze.{entity_name}")
    old_pools_table.alias("old").merge(
        pools_df.alias("new"),
        "old.id = new.id AND old.token0.id = new.token0.id AND old.token1.id = new.token1.id"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")
else:
    print("*ERROR*: Invalid load pattern.")
    dbutils.notebook.exit("Invalid load pattern.")